# Notebook9 parameter finalization — staged v9 package

Existing private Kaggle identity `satyapaladugu/betterscheduler-spi-20261027finalc` (ID 141089219), version 9. This package enables real execution only in the deployed runtime config. It requires two T4 GPUs and caps server time at 7,200 seconds (4 aggregate GPU-hours). Source snapshot and separately embedded runtime config are SHA-256 verified. The attached 300-image development input is hash checked; first-image HF CLIP feature-signature equality and actual 16-class YOLO-WORLD inference must pass before any Qwen labeling. No synthetic run substitutes for real data.

Expected outputs: `/kaggle/working/experiments/parameter_finalization/runs/notebook9_20261028/`, incremental per-stage logs/status and `executed.ipynb`, plus `/kaggle/working/run_recovery.zip`. A `finally` block refreshes recovery after success or failure. Full run remains subject to cache readiness gates; teacher labels are VLM pseudo-labels, not independent urgency truth.

In [ ]:
import os,sys,json,base64,hashlib,zipfile,io,subprocess,time,shutil,urllib.request
from pathlib import Path
print('START immutable Notebook9 package bootstrap',flush=True)
PKG=Path('/kaggle/working/notebook9_package'); ROOT=Path('/kaggle/working/project')
PKG.mkdir(exist_ok=False); ROOT.mkdir(exist_ok=False)
SOURCE_BYTES=base64.b64decode('...')
if hashlib.sha256(SOURCE_BYTES).hexdigest()!='560920182dedba6bdb3e81c00e3f2fb7df130389d5e8b945040631934da73c7c': raise RuntimeError('embedded source archive checksum mismatch')
with zipfile.ZipFile(io.BytesIO(SOURCE_BYTES)) as z:
    if any(not (ROOT/name).resolve().is_relative_to(ROOT) for name in z.namelist()): raise ValueError('unsafe source path')
    z.extractall(ROOT)
SOURCE=json.loads((ROOT/'SOURCE_MANIFEST.json').read_text())
for name,digest in SOURCE['files'].items():
    path=(ROOT/name).resolve()
    if not path.is_relative_to(ROOT) or hashlib.sha256(path.read_bytes()).hexdigest()!=digest: raise RuntimeError('source hash mismatch: '+name)
CONFIG_BYTES=base64.b64decode('eyJjbGlwX2NoZWNrcG9pbnQiOiIva2FnZ2xlL3dvcmtpbmcvcHJvamVjdC9tb2RlbHMvcHJpb3JpdHlfY2xhc3NpZmllcnMvY2xpcCIsImRldGVjdG9yIjoieW9sb193b3JsZCIsImRldGVjdG9yX2NoZWNrcG9pbnQiOiIva2FnZ2xlL3dvcmtpbmcvcHJvamVjdC9tb2RlbHMveW9sb3Y4cy13b3JsZHYyLnB0IiwiZGV2aWNlIjoiY3VkYTowIiwiaW5wdXRfcm9vdCI6Ii9rYWdnbGUvaW5wdXQvYmV0dGVyc2NoZWR1bGVyLXNwaXYyLWlucHV0cy0yMDI2MTAyN2IifQ==')
if hashlib.sha256(CONFIG_BYTES).hexdigest()!='954df05f24610823e651ee33665f75527ce254711b79d99f0d27aaa10c74bb57': raise RuntimeError('runtime config hash mismatch')
CONFIG_PATH=Path('/kaggle/working/notebook9_config.json'); CONFIG_PATH.write_bytes(CONFIG_BYTES)
os.environ['SEMANTIC_PROJECT_ROOT']=str(ROOT);os.environ['PARAMETER_RUN_REAL']='1'
os.environ['PARAMETER_RUN_ID']='20261028';os.environ['PARAMETER_CONFIG']=str(CONFIG_PATH)
os.environ['PARAMETER_OUTPUT_ROOT']='/kaggle/working/experiments/parameter_finalization/runs/notebook9_20261028/data_run'
CONFIG=json.loads(CONFIG_BYTES)
os.environ['YOLO_CONFIG_DIR']='/kaggle/temp/notebook9_ultralytics'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT']='300'
for name in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS'):os.environ[name]='1'
sys.path.insert(0,str(ROOT))
CELL_SOURCES=["import os,sys,json,time,hashlib,subprocess,importlib.metadata as metadata\nfrom pathlib import Path\nprint('START notebook9 setup',flush=True)\nROOT=Path(os.environ.get('SEMANTIC_PROJECT_ROOT',Path.cwd())).resolve()\nif not (ROOT/'parameters/dataset_pipeline.py').is_file():raise RuntimeError('Set SEMANTIC_PROJECT_ROOT to canonical project checkout')\nsys.path.insert(0,str(ROOT))\nprint('Repository',ROOT,'Python',sys.executable,sys.version,flush=True)\ntry: revision=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True,stderr=subprocess.DEVNULL).strip()\nexcept (subprocess.CalledProcessError,FileNotFoundError): revision='packaged snapshot; inspect source_manifest.json'\nprint('Revision',revision,flush=True)\nprint('Packages',{p:metadata.version(p) for p in ('numpy','Pillow','nbformat')},flush=True)\ntry:\n import torch\n print('Torch',torch.__version__,'CUDA',torch.version.cuda,'GPUs',[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],flush=True)\nexcept ImportError:print('Torch unavailable; real neural path blocked')\nRUN_ID=os.environ.get('PARAMETER_RUN_ID',time.strftime('%Y%m%d_%H%M%S'))\nOUTPUT=Path(os.environ.get('PARAMETER_OUTPUT_ROOT',str(ROOT/'experiments/parameter_finalization/runs'/('notebook9_'+RUN_ID))))\nOUTPUT.mkdir(parents=True,exist_ok=False)\nRUN_REAL=False  # Safe local default\nif os.environ.get('PARAMETER_RUN_REAL') == '1': RUN_REAL=True\nprint('Output',OUTPUT,'RUN_REAL',RUN_REAL,flush=True)\nprint('END notebook9 setup',flush=True)\n", "print('START fixture checks',flush=True)\nif not RUN_REAL:\n from experiments.parameter_finalization.test_dataset_pipeline import run_dummy\n DUMMY_OUTPUT=run_dummy(OUTPUT/'dummy')\n print('Synthetic software fixture only',json.loads((DUMMY_OUTPUT/'status.json').read_text()),flush=True)\nelse: print('Real run: synthetic walkthrough skipped',flush=True)\nprint('END fixture checks',flush=True)\n", "print('START final300 configuration',flush=True)\nCONFIG_PATH=Path(os.environ.get('PARAMETER_CONFIG','notebook9_config.json'))\nCONFIG=json.loads(CONFIG_PATH.read_text()) if CONFIG_PATH.is_file() else None\nTEACHER_RECEIPT_PATH=ROOT/'parameters/config/final_labeling_teacher.json'\nTEACHER_RECEIPT=json.loads(TEACHER_RECEIPT_PATH.read_text())\nprint('Final teacher',TEACHER_RECEIPT['model_id'],TEACHER_RECEIPT['revision'],flush=True)\nif RUN_REAL and CONFIG is None:raise RuntimeError('Missing image/checkpoint configuration')\nif CONFIG is not None:\n CONFIG.setdefault('detector','yolo_world')\n if CONFIG['detector'] not in ('yolo_world','grounding_dino'):raise ValueError('Select one detector, no ensemble')\nprint('Spatial interpretation: detection counts/category diversity/rectangular extent; not unique assets, distance or urgency',flush=True)\nprint('Real execution',RUN_REAL,'Default detector','yolo_world',flush=True)\nprint('Required config: input_root (existing300 bundle) OR records_path+features_path, clip_checkpoint, detector_checkpoint; optional device/detector',flush=True)\nprint('END final300 configuration',flush=True)\n", "print('START Qwen3.5 labeling of300 images',flush=True)\nif RUN_REAL:\n from parameters.dataset_pipeline import label_dataset,export_spi_training,load_selected_teacher,materialize_current_300\n if CONFIG.get('input_root') and CONFIG.get('records_path'):raise ValueError('Choose input_root or records_path, not both')\n if CONFIG.get('input_root'):\n  records,features=materialize_current_300(CONFIG['input_root'],OUTPUT/'input_images')\n  CONFIG['features_path']=str(features)\n else:records=json.loads(Path(CONFIG['records_path']).read_text())\n if len(records)!=300:raise ValueError('This milestone requires exactly300 images')\n # Fail early on wrong feature space or broken detector before 300 VLM calls.\n from parameters.minimal_vision import MinimalVisionProvider\n from parameters.contracts import MissionSpec\n from parameters.extractor import embedding_signature\n from parameters.spatial_provider import make_spatial_provider\n from PIL import Image\n import numpy as np,gc,torch\n preflight_mission=MissionSpec('preflight','Describe visible evidence.',(('visible','a photograph'),))\n preflight_provider=MinimalVisionProvider(CONFIG['clip_checkpoint'],{'preflight':preflight_mission},device=CONFIG.get('device','cpu'))\n preflight_scene=make_spatial_provider(CONFIG['detector_checkpoint'],detector=CONFIG['detector'],device=CONFIG.get('device','cpu'))\n try:\n  with Image.open(records[0]['image_path']) as im: preflight_image=im.copy()\n  preflight_evidence=preflight_provider.extract_batch([preflight_image],[preflight_mission])[0]\n  with np.load(CONFIG['features_path'],allow_pickle=False) as archive: expected_signature=str(archive['feature_signature'].item())\n  if embedding_signature(preflight_evidence)!=expected_signature:raise RuntimeError('Fresh CLIP producer differs from cached training feature space')\n  preflight_measurements=preflight_scene.measure(preflight_image)\n  if preflight_measurements.status!='ok':raise RuntimeError('Detector preflight failed: '+str(preflight_measurements.reason))\n  from parameters.artifacts import write_json\n  write_json(OUTPUT/'producer_preflight.json',{'feature_signature':expected_signature,'scene':preflight_measurements.audit,'status':'passed'})\n finally:\n  preflight_provider.close()\n  del preflight_provider,preflight_scene\n  gc.collect()\n  if torch.cuda.is_available():torch.cuda.empty_cache()\n print('Producer preflight passed; starting accepted teacher',flush=True)\n teacher=load_selected_teacher(TEACHER_RECEIPT)\n try:label_dataset(records,teacher,TEACHER_RECEIPT,OUTPUT/'teacher_labels')\n finally:\n  teacher.close()\n  del teacher\n  gc.collect()\n  if torch.cuda.is_available():torch.cuda.empty_cache()\n export_spi_training(CONFIG['features_path'],OUTPUT/'teacher_labels/labels.json',OUTPUT/'spi_training')\n print('Teacher labels and training bundle saved',flush=True)\nelse:print('Prepared only: no real labeling launched',flush=True)\nprint('END Qwen3.5 labeling stage',flush=True)\n", "print('START four-class SPI classifier training on300-image cohort',flush=True)\nif RUN_REAL:\n from parameters.dataset_pipeline import train_spi_for_cache\n from parameters.artifacts import write_json\n TRAINING=train_spi_for_cache(OUTPUT/'spi_training/features.npz',OUTPUT/'spi_training/labels.json',\n   TEACHER_RECEIPT,OUTPUT/'spi_classifier',ROOT/'parameters/config/parameter_release_v2.json',root=ROOT,seconds=600)\n write_json(OUTPUT/'spi_training_result.json',TRAINING)\n print('Student validation',TRAINING['validation'],flush=True)\n print('Parameter cache permitted',TRAINING['cache_allowed'],flush=True)\n if not TRAINING['cache_allowed']:raise RuntimeError('Student support/recall/export checks failed; preserve model, stop before cache')\n CONFIG['model_path']=TRAINING['model_path'];CONFIG['readiness_path']=TRAINING['readiness_path']\nelse:print('Prepared only: no real SPI classifier trained',flush=True)\nprint('END four-class SPI training',flush=True)\n", "print('START readiness-gated parameter cache',flush=True)\nif RUN_REAL:\n from parameters.readiness import require_spi_ready\n from parameters.dataset_pipeline import cache_parameters\n from parameters.minimal_vision import MinimalVisionProvider\n from parameters.contracts import MissionSpec\n from parameters.spatial_provider import make_spatial_provider\n require_spi_ready(CONFIG['model_path'],CONFIG['readiness_path'])\n p=json.loads((ROOT/'parameters/config/ue_parameter_profiles_v2.json').read_text())\n profiles=[(r['ue_id'],MissionSpec(r['mission_id'],r['paragraph'],((r['mission_id'],r['paragraph']),),allowed_delay_ms=r['allowed_delay_ms'])) for r in p['profiles']]\n provider=MinimalVisionProvider(CONFIG['clip_checkpoint'],{m.mission_id:m for _,m in profiles},device=CONFIG.get('device','cpu'))\n scene=make_spatial_provider(CONFIG['detector_checkpoint'],detector=CONFIG.get('detector','yolo_world'),device=CONFIG.get('device','cpu'))\n try:cache_parameters(records,profiles,provider,scene,CONFIG['model_path'],CONFIG['readiness_path'],OUTPUT/'parameter_cache')\n finally:\n  provider.close()\n  if hasattr(scene,'close'):scene.close()\n  del scene,provider\n  gc.collect()\n  if torch.cuda.is_available():torch.cuda.empty_cache()\nelse:print('BLOCKED: real four-class SPI readiness is pending; dummy cache is separate.',flush=True)\nprint('END parameter-cache stage',flush=True)\n", "print('START full-corpus resource gate',flush=True)\nprint('Full sources: LADI-v2, MEDIC, CrisisMMD and legitimately linked TREC-IS images.',flush=True)\nprint('Do not invent inaccessible assets or image urgency labels from post priority.',flush=True)\nprint('Required: deduplicated grouped manifest, source licenses/revisions/native splits, projected bytes and GPU hours; <=20GB data/eight aggregate GPU hours.',flush=True)\nprint('Full-corpus extraction is NOT launched by this notebook before the reviewed300 milestone and resource receipt.',flush=True)\nprint('Timing: per-stage calls and cold-first status. FLOPs: null unless separately supported-operator profiled; not complete totals.',flush=True)\nprint('END full-corpus resource gate',flush=True)\n"]
from parameters.cloud_notebook import CellRecorder
INTERNAL_CELLS=['dependency and two-GPU preflight plus pinned model assets',*CELL_SOURCES[:3],
 'input-bundle verification, CLIP signature equality and real YOLO-WORLD warmup',*CELL_SOURCES[3:]]
RECORDER=CellRecorder(INTERNAL_CELLS,Path('/kaggle/working/experiments/parameter_finalization/runs/notebook9_20261028/cell_evidence'))
print('Verified source files',len(SOURCE['files']),'source SHA-256','560920182dedba6bdb3e81c00e3f2fb7df130389d5e8b945040631934da73c7c','config SHA-256','954df05f24610823e651ee33665f75527ce254711b79d99f0d27aaa10c74bb57',flush=True)
print('END immutable Notebook9 package bootstrap',flush=True)


In [ ]:
DEPENDENCY_PREFLIGHT = "import importlib.metadata as md, torch, gc, hashlib, json, shutil, subprocess, urllib.request, time, zipfile\nfrom pathlib import Path\nprint('START dependency, hardware, model-asset preflight',flush=True)\ncommands=[\n [sys.executable,'-m','pip','install','--disable-pip-version-check','setuptools==80.9.0'],\n [sys.executable,'-m','pip','install','--disable-pip-version-check','transformers==5.5.0','huggingface-hub==1.5.0','accelerate==1.12.0','bitsandbytes==0.49.2','sentencepiece==0.2.2','scikit-learn==1.7.2','nbformat==5.10.4','ftfy==6.3.1','opencv-python==4.11.0.86','polars==1.37.1','polars-runtime-32==1.37.1','ultralytics-thop==2.0.18'],\n [sys.executable,'-m','pip','install','--disable-pip-version-check','--no-deps','ultralytics==8.3.237'],\n [sys.executable,'-m','pip','install','--disable-pip-version-check','--no-deps','--no-build-isolation','git+https://github.com/openai/CLIP.git@a1d071733d7111c9c014f024669f959182114e33']]\nfor command in commands:\n print('RUN',command,flush=True);result=subprocess.run(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=600);print(result.stdout,flush=True);result.check_returncode()\nimport torch\nif torch.cuda.device_count()!=2: raise RuntimeError('Notebook9 requires exactly two allocated GPUs before downloading or inference')\nGPU_NAMES=[torch.cuda.get_device_name(i) for i in range(2)]\nif any('T4' not in name for name in GPU_NAMES): raise RuntimeError('Runtime must match reviewed two-T4 resource contract: '+repr(GPU_NAMES))\nprint('Actual interpreter',sys.executable,sys.version,'torch',torch.__version__,'CUDA',torch.version.cuda,'GPUs',GPU_NAMES,flush=True)\n\ndef download(url,destination,expected_sha,expected_bytes,timeout=90):\n    destination=Path(destination);destination.parent.mkdir(parents=True,exist_ok=True);part=destination.with_suffix(destination.suffix+'.partial')\n    h=hashlib.sha256();count=0;started=time.monotonic()\n    request=urllib.request.Request(url,headers={'User-Agent':'Notebook9-parameter-finalization/1'})\n    with urllib.request.urlopen(request,timeout=timeout) as response,part.open('wb') as out:\n        while True:\n            block=response.read(1024*1024)\n            if not block: break\n            count+=len(block)\n            if count>expected_bytes: raise RuntimeError('asset exceeds declared byte cap')\n            h.update(block);out.write(block);out.flush()\n    if count!=expected_bytes or h.hexdigest()!=expected_sha: raise RuntimeError('downloaded asset size/hash mismatch: '+str(destination))\n    part.replace(destination);print('Verified asset',destination,'bytes',count,'sha256',h.hexdigest(),'seconds',round(time.monotonic()-started,2),flush=True)\n\n# Download only the pinned HF model tensor; config/processor/tokenizer are source-archive members.\nfrom huggingface_hub import hf_hub_download\nclip_root=Path(CONFIG['clip_checkpoint']);clip_root.mkdir(parents=True,exist_ok=True)\nfor rel in ('config.json','preprocessor_config.json','tokenizer_config.json','special_tokens_map.json'):\n    if not (clip_root/rel).is_file(): raise RuntimeError('source archive missing HF CLIP file: '+rel)\nREMOTE_CLIP_FILES={'tokenizer.json':(2224041,'b556ac8c99757ffb677208af34bc8c6721572111a6e0aaf5fa69ff0b8d842'),'vocab.json':(862328,'5047b556ce86ccaf6aa22b3ffccfc52d391ea4accdab9c2f2407da5b742d4363'),'merges.txt':(524657,'f526393189112391ce6f9795d4695f704121ce452c3aad1f5335cc41337eba85')}\nfor name,(size,expected) in REMOTE_CLIP_FILES.items():\n    downloaded=Path(hf_hub_download(repo_id='openai/clip-vit-base-patch32',filename=name,revision='3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268'))\n    if downloaded.stat().st_size!=size or hashlib.sha256(downloaded.read_bytes()).hexdigest()!=expected: raise RuntimeError('pinned HF CLIP tokenizer asset size/hash mismatch: '+name)\n    shutil.copy2(downloaded,clip_root/name)\nweight=hf_hub_download(repo_id='openai/clip-vit-base-patch32',filename='pytorch_model.bin',revision='3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268')\nweight_dst=clip_root/'pytorch_model.bin';shutil.copy2(weight,weight_dst)\nif hashlib.sha256(weight_dst.read_bytes()).hexdigest()!='a63082132ba4f97a80bea76823f544493bffa8082296d62d71581a4feff1576f':raise RuntimeError('pinned HF CLIP weight hash differs from locally verified feature producer')\ndownload('https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8s-worldv2.pt',CONFIG['detector_checkpoint'],'9b2c17ab6124a913e9b3a5c170617920d91b0f01111a8479da69f00e2cf27792',25923032)\nopenai_cache=Path.home()/'.cache/clip/ViT-B-32.pt'\ndownload('https://openaipublic.azureedge.net/clip/models/40d365715913c9da98579312b702a82c18be219cc2a73407c4526f58eba950af/ViT-B-32.pt',openai_cache,'40d365715913c9da98579312b702a82c18be219cc2a73407c4526f58eba950af',353976522)\nDEPENDENCIES={p:md.version(p) for p in ('torch','numpy','transformers','huggingface-hub','accelerate','bitsandbytes','scikit-learn','nbformat','ultralytics','setuptools','ftfy')}\nRUNTIME={'python':sys.version,'interpreter':sys.executable,'packages':DEPENDENCIES,'cuda':torch.version.cuda,'gpus':GPU_NAMES,'source_sha256':hashlib.sha256(SOURCE_BYTES).hexdigest(),'runtime_config_sha256':hashlib.sha256(CONFIG_BYTES).hexdigest(),'run_id':'notebook9_20261028'}\nPath('/kaggle/working/notebook9_runtime.json').write_text(json.dumps(RUNTIME,indent=2))\nprint('END dependencies and pinned assets preflight',flush=True)\n"
INPUT_PREFLIGHT = "import json,zipfile,hashlib,gc,time\nfrom PIL import Image\nfrom parameters.spi_campaign import verify_inputs\nfrom parameters.contracts import MissionSpec\nfrom parameters.extractor import embedding_signature\nfrom parameters.minimal_vision import MinimalVisionProvider\nfrom parameters.spatial_provider import make_spatial_provider\nINPUT=Path(CONFIG['input_root'])\nrows,INPUT_RECEIPT=verify_inputs(INPUT)\nif len(rows)!=300: raise RuntimeError('Existing input must contain exactly 300 verified rows')\nmanifest=json.loads((INPUT/'input_manifest.json').read_text())\nexpected=manifest['feature_signature']\nprofile=json.loads((ROOT/'parameters/config/ue_parameter_profiles_v2.json').read_text())\nif profile['feature_signature']!=expected: raise RuntimeError('profile/input feature signature mismatch')\nrow=rows[0]\nif INPUT_RECEIPT['layout']=='zip':\n    with zipfile.ZipFile(INPUT/'images.zip') as z: raw=z.read(row['image_path'])\n    from io import BytesIO\n    image=Image.open(BytesIO(raw)).convert('RGB')\nelse:\n    image_path=INPUT/'images'/row['image_path']\n    if hashlib.sha256(image_path.read_bytes()).hexdigest()!=row['image_sha256']: raise RuntimeError('first-image SHA mismatch')\n    image=Image.open(image_path).convert('RGB')\nfirst_profile=profile['profiles'][0]\nmission=MissionSpec(first_profile['mission_id'],first_profile['paragraph'],((first_profile['mission_id'],first_profile['paragraph']),),allowed_delay_ms=first_profile['allowed_delay_ms'])\nvision=MinimalVisionProvider(CONFIG['clip_checkpoint'],{mission.mission_id:mission},device='cuda:0')\ntry:\n    vision.provider.load()\n    evidence=vision.extract_batch([image],[mission])[0]\n    measured=embedding_signature(evidence)\n    print('Input layout',INPUT_RECEIPT,'CLIP signature',measured,'expected',expected,flush=True)\n    if measured!=expected: raise RuntimeError('fresh CLIP embedding signature differs from existing 300-image features; stop before teacher labeling')\nfinally: vision.close();del vision;gc.collect();torch.cuda.empty_cache()\nscene=make_spatial_provider(CONFIG['detector_checkpoint'],detector='yolo_world',device='cuda:0')\ntry:\n    observation=scene.measure(image)\n    print('YOLO-WORLD vocabulary',len(scene.vocabulary),'status',observation.status,'reason',observation.reason,'measurements',observation.values(),flush=True)\n    if observation.status!='ok': raise RuntimeError('real YOLO-WORLD first-image preflight failed; stop before teacher labeling')\n    if len(scene.vocabulary)!=16: raise RuntimeError('runtime vocabulary changed from declared 16-class specification')\nfinally:\n    if hasattr(scene,'close'): scene.close()\n    del scene;gc.collect();torch.cuda.empty_cache()\nPRECHECK={'input_images':len(rows),'input_manifest_sha256':hashlib.sha256((INPUT/'input_manifest.json').read_bytes()).hexdigest(),'feature_signature':measured,'first_image_sha256':row['image_sha256'],'detector':'yolo_world','detector_status':observation.status,'detector_vocabulary_count':16,'gpus':GPU_NAMES,'status':'passed'}\nPath('/kaggle/working/notebook9_prelabel_preflight.json').write_text(json.dumps(PRECHECK,indent=2))\nprint('END pre-label real feature/detector checks; teacher cell may now start',flush=True)\n"
CELL_SOURCES = ["import os,sys,json,time,hashlib,subprocess,importlib.metadata as metadata\nfrom pathlib import Path\nprint('START notebook9 setup',flush=True)\nROOT=Path(os.environ.get('SEMANTIC_PROJECT_ROOT',Path.cwd())).resolve()\nif not (ROOT/'parameters/dataset_pipeline.py').is_file():raise RuntimeError('Set SEMANTIC_PROJECT_ROOT to canonical project checkout')\nsys.path.insert(0,str(ROOT))\nprint('Repository',ROOT,'Python',sys.executable,sys.version,flush=True)\ntry: revision=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True,stderr=subprocess.DEVNULL).strip()\nexcept (subprocess.CalledProcessError,FileNotFoundError): revision='packaged snapshot; inspect source_manifest.json'\nprint('Revision',revision,flush=True)\nprint('Packages',{p:metadata.version(p) for p in ('numpy','Pillow','nbformat')},flush=True)\ntry:\n import torch\n print('Torch',torch.__version__,'CUDA',torch.version.cuda,'GPUs',[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],flush=True)\nexcept ImportError:print('Torch unavailable; real neural path blocked')\nRUN_ID=os.environ.get('PARAMETER_RUN_ID',time.strftime('%Y%m%d_%H%M%S'))\nOUTPUT=Path(os.environ.get('PARAMETER_OUTPUT_ROOT',str(ROOT/'experiments/parameter_finalization/runs'/('notebook9_'+RUN_ID))))\nOUTPUT.mkdir(parents=True,exist_ok=False)\nRUN_REAL=False  # Safe local default\nif os.environ.get('PARAMETER_RUN_REAL') == '1': RUN_REAL=True\nprint('Output',OUTPUT,'RUN_REAL',RUN_REAL,flush=True)\nprint('END notebook9 setup',flush=True)\n", "print('START fixture checks',flush=True)\nif not RUN_REAL:\n from experiments.parameter_finalization.test_dataset_pipeline import run_dummy\n DUMMY_OUTPUT=run_dummy(OUTPUT/'dummy')\n print('Synthetic software fixture only',json.loads((DUMMY_OUTPUT/'status.json').read_text()),flush=True)\nelse: print('Real run: synthetic walkthrough skipped',flush=True)\nprint('END fixture checks',flush=True)\n", "print('START final300 configuration',flush=True)\nCONFIG_PATH=Path(os.environ.get('PARAMETER_CONFIG','notebook9_config.json'))\nCONFIG=json.loads(CONFIG_PATH.read_text()) if CONFIG_PATH.is_file() else None\nTEACHER_RECEIPT_PATH=ROOT/'parameters/config/final_labeling_teacher.json'\nTEACHER_RECEIPT=json.loads(TEACHER_RECEIPT_PATH.read_text())\nprint('Final teacher',TEACHER_RECEIPT['model_id'],TEACHER_RECEIPT['revision'],flush=True)\nif RUN_REAL and CONFIG is None:raise RuntimeError('Missing image/checkpoint configuration')\nif CONFIG is not None:\n CONFIG.setdefault('detector','yolo_world')\n if CONFIG['detector'] not in ('yolo_world','grounding_dino'):raise ValueError('Select one detector, no ensemble')\nprint('Spatial interpretation: detection counts/category diversity/rectangular extent; not unique assets, distance or urgency',flush=True)\nprint('Real execution',RUN_REAL,'Default detector','yolo_world',flush=True)\nprint('Required config: input_root (existing300 bundle) OR records_path+features_path, clip_checkpoint, detector_checkpoint; optional device/detector',flush=True)\nprint('END final300 configuration',flush=True)\n", "print('START Qwen3.5 labeling of300 images',flush=True)\nif RUN_REAL:\n from parameters.dataset_pipeline import label_dataset,export_spi_training,load_selected_teacher,materialize_current_300\n if CONFIG.get('input_root') and CONFIG.get('records_path'):raise ValueError('Choose input_root or records_path, not both')\n if CONFIG.get('input_root'):\n  records,features=materialize_current_300(CONFIG['input_root'],OUTPUT/'input_images')\n  CONFIG['features_path']=str(features)\n else:records=json.loads(Path(CONFIG['records_path']).read_text())\n if len(records)!=300:raise ValueError('This milestone requires exactly300 images')\n # Fail early on wrong feature space or broken detector before 300 VLM calls.\n from parameters.minimal_vision import MinimalVisionProvider\n from parameters.contracts import MissionSpec\n from parameters.extractor import embedding_signature\n from parameters.spatial_provider import make_spatial_provider\n from PIL import Image\n import numpy as np,gc,torch\n preflight_mission=MissionSpec('preflight','Describe visible evidence.',(('visible','a photograph'),))\n preflight_provider=MinimalVisionProvider(CONFIG['clip_checkpoint'],{'preflight':preflight_mission},device=CONFIG.get('device','cpu'))\n preflight_scene=make_spatial_provider(CONFIG['detector_checkpoint'],detector=CONFIG['detector'],device=CONFIG.get('device','cpu'))\n try:\n  with Image.open(records[0]['image_path']) as im: preflight_image=im.copy()\n  preflight_evidence=preflight_provider.extract_batch([preflight_image],[preflight_mission])[0]\n  with np.load(CONFIG['features_path'],allow_pickle=False) as archive: expected_signature=str(archive['feature_signature'].item())\n  if embedding_signature(preflight_evidence)!=expected_signature:raise RuntimeError('Fresh CLIP producer differs from cached training feature space')\n  preflight_measurements=preflight_scene.measure(preflight_image)\n  if preflight_measurements.status!='ok':raise RuntimeError('Detector preflight failed: '+str(preflight_measurements.reason))\n  from parameters.artifacts import write_json\n  write_json(OUTPUT/'producer_preflight.json',{'feature_signature':expected_signature,'scene':preflight_measurements.audit,'status':'passed'})\n finally:\n  preflight_provider.close()\n  del preflight_provider,preflight_scene\n  gc.collect()\n  if torch.cuda.is_available():torch.cuda.empty_cache()\n print('Producer preflight passed; starting accepted teacher',flush=True)\n teacher=load_selected_teacher(TEACHER_RECEIPT)\n try:label_dataset(records,teacher,TEACHER_RECEIPT,OUTPUT/'teacher_labels')\n finally:\n  teacher.close()\n  del teacher\n  gc.collect()\n  if torch.cuda.is_available():torch.cuda.empty_cache()\n export_spi_training(CONFIG['features_path'],OUTPUT/'teacher_labels/labels.json',OUTPUT/'spi_training')\n print('Teacher labels and training bundle saved',flush=True)\nelse:print('Prepared only: no real labeling launched',flush=True)\nprint('END Qwen3.5 labeling stage',flush=True)\n", "print('START four-class SPI classifier training on300-image cohort',flush=True)\nif RUN_REAL:\n from parameters.dataset_pipeline import train_spi_for_cache\n from parameters.artifacts import write_json\n TRAINING=train_spi_for_cache(OUTPUT/'spi_training/features.npz',OUTPUT/'spi_training/labels.json',\n   TEACHER_RECEIPT,OUTPUT/'spi_classifier',ROOT/'parameters/config/parameter_release_v2.json',root=ROOT,seconds=600)\n write_json(OUTPUT/'spi_training_result.json',TRAINING)\n print('Student validation',TRAINING['validation'],flush=True)\n print('Parameter cache permitted',TRAINING['cache_allowed'],flush=True)\n if not TRAINING['cache_allowed']:raise RuntimeError('Student support/recall/export checks failed; preserve model, stop before cache')\n CONFIG['model_path']=TRAINING['model_path'];CONFIG['readiness_path']=TRAINING['readiness_path']\nelse:print('Prepared only: no real SPI classifier trained',flush=True)\nprint('END four-class SPI training',flush=True)\n", "print('START readiness-gated parameter cache',flush=True)\nif RUN_REAL:\n from parameters.readiness import require_spi_ready\n from parameters.dataset_pipeline import cache_parameters\n from parameters.minimal_vision import MinimalVisionProvider\n from parameters.contracts import MissionSpec\n from parameters.spatial_provider import make_spatial_provider\n require_spi_ready(CONFIG['model_path'],CONFIG['readiness_path'])\n p=json.loads((ROOT/'parameters/config/ue_parameter_profiles_v2.json').read_text())\n profiles=[(r['ue_id'],MissionSpec(r['mission_id'],r['paragraph'],((r['mission_id'],r['paragraph']),),allowed_delay_ms=r['allowed_delay_ms'])) for r in p['profiles']]\n provider=MinimalVisionProvider(CONFIG['clip_checkpoint'],{m.mission_id:m for _,m in profiles},device=CONFIG.get('device','cpu'))\n scene=make_spatial_provider(CONFIG['detector_checkpoint'],detector=CONFIG.get('detector','yolo_world'),device=CONFIG.get('device','cpu'))\n try:cache_parameters(records,profiles,provider,scene,CONFIG['model_path'],CONFIG['readiness_path'],OUTPUT/'parameter_cache')\n finally:\n  provider.close()\n  if hasattr(scene,'close'):scene.close()\n  del scene,provider\n  gc.collect()\n  if torch.cuda.is_available():torch.cuda.empty_cache()\nelse:print('BLOCKED: real four-class SPI readiness is pending; dummy cache is separate.',flush=True)\nprint('END parameter-cache stage',flush=True)\n", "print('START full-corpus resource gate',flush=True)\nprint('Full sources: LADI-v2, MEDIC, CrisisMMD and legitimately linked TREC-IS images.',flush=True)\nprint('Do not invent inaccessible assets or image urgency labels from post priority.',flush=True)\nprint('Required: deduplicated grouped manifest, source licenses/revisions/native splits, projected bytes and GPU hours; <=20GB data/eight aggregate GPU hours.',flush=True)\nprint('Full-corpus extraction is NOT launched by this notebook before the reviewed300 milestone and resource receipt.',flush=True)\nprint('Timing: per-stage calls and cold-first status. FLOPs: null unless separately supported-operator profiled; not complete totals.',flush=True)\nprint('END full-corpus resource gate',flush=True)\n"]
print("START Notebook9 staged run",flush=True)
run_failed=None
try:
    with RECORDER.cell(0):
        exec(compile(DEPENDENCY_PREFLIGHT, "dependency_preflight", "exec"), globals())
    with RECORDER.cell(1):
        exec(compile(CELL_SOURCES[0], "notebook9_source_cell_0", "exec"), globals())
    with RECORDER.cell(2):
        exec(compile(CELL_SOURCES[1], "notebook9_source_cell_1", "exec"), globals())
    with RECORDER.cell(3):
        exec(compile(CELL_SOURCES[2], "notebook9_source_cell_2", "exec"), globals())
    with RECORDER.cell(4):
        exec(compile(INPUT_PREFLIGHT, "input_detector_preflight", "exec"), globals())
    with RECORDER.cell(5):
        exec(compile(CELL_SOURCES[3], "notebook9_source_cell_3", "exec"), globals())
    with RECORDER.cell(6):
        exec(compile(CELL_SOURCES[4], "notebook9_source_cell_4", "exec"), globals())
    with RECORDER.cell(7):
        exec(compile(CELL_SOURCES[5], "notebook9_source_cell_5", "exec"), globals())
    with RECORDER.cell(8):
        exec(compile(CELL_SOURCES[6], "notebook9_source_cell_6", "exec"), globals())
except BaseException as error:
    run_failed=error; print("NOTEBOOK9 RUN FAILED; preserving current outputs",type(error).__name__,str(error),flush=True)
finally:
    run_root=Path("/kaggle/working/experiments/parameter_finalization/runs/notebook9_20261028")
    run_root.mkdir(parents=True,exist_ok=True)
    for src,dst in ((Path("/kaggle/working/notebook9_runtime.json"),run_root/"runtime.json"),(Path("/kaggle/working/notebook9_prelabel_preflight.json"),run_root/"prelabel_preflight.json"),(Path("/kaggle/working/notebook9_config.json"),run_root/"runtime_config.json"),(ROOT/"SOURCE_MANIFEST.json",run_root/"SOURCE_MANIFEST.json")):
        if src.exists(): shutil.copy2(src,dst)
    if run_root.exists():
        payload=[p for p in run_root.rglob("*") if p.is_file() and p.name not in ("retrieval_checksums.json","run_recovery.zip")]
        checks={str(p.relative_to(run_root)):hashlib.sha256(p.read_bytes()).hexdigest() for p in payload}
        (run_root/"retrieval_checksums.json").write_text(json.dumps(checks,indent=2))
        with zipfile.ZipFile("/kaggle/working/run_recovery.zip","w",zipfile.ZIP_DEFLATED) as recovery:
            for p in payload: recovery.write(p,str(p.relative_to(run_root)))
            recovery.write(run_root/"retrieval_checksums.json","retrieval_checksums.json")
    print("Recovery archive updated; files",len(payload),"failed",repr(run_failed),flush=True)
if run_failed is not None: raise run_failed
print("END Notebook9 staged run",flush=True)